# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [24]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [25]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1145


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [26]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [27]:
# YOUR CHANGES HERE

import csv

with open("log-random.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["curr_state", "curr_action", "next_reward", "next_state"])
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
        print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        writer.writerow([curr_state, curr_action, next_reward, next_state])

CURR STATE 66 ACTION 1 NEXT REWARD 0 NEXT STATE 75
CURR STATE 75 ACTION -1 NEXT REWARD 0 NEXT STATE 82
CURR STATE 82 ACTION 1 NEXT REWARD 0 NEXT STATE 91
CURR STATE 91 ACTION 0 NEXT REWARD 0 NEXT STATE 99
CURR STATE 99 ACTION -1 NEXT REWARD 0 NEXT STATE 106
CURR STATE 106 ACTION 1 NEXT REWARD 0 NEXT STATE 115
CURR STATE 115 ACTION 1 NEXT REWARD 0 NEXT STATE 124
CURR STATE 124 ACTION 1 NEXT REWARD 0 NEXT STATE 125
CURR STATE 125 ACTION 1 NEXT REWARD 0 NEXT STATE 118
CURR STATE 118 ACTION -1 NEXT REWARD 0 NEXT STATE 109
CURR STATE 109 ACTION 0 NEXT REWARD 0 NEXT STATE 101
CURR STATE 101 ACTION 1 NEXT REWARD 0 NEXT STATE 94
CURR STATE 94 ACTION -1 NEXT REWARD 0 NEXT STATE 85
CURR STATE 85 ACTION 0 NEXT REWARD 0 NEXT STATE 77
CURR STATE 77 ACTION 0 NEXT REWARD 0 NEXT STATE 69
CURR STATE 69 ACTION 1 NEXT REWARD 0 NEXT STATE 62
CURR STATE 62 ACTION 0 NEXT REWARD 0 NEXT STATE 54
CURR STATE 54 ACTION 1 NEXT REWARD 0 NEXT STATE 47
CURR STATE 47 ACTION 1 NEXT REWARD 0 NEXT STATE 160
CURR STATE 1

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [28]:
# YOUR CHANGES HERE

ALPHA = 1.0
GAMMA = 0.9
q = {}

def get_q(state, action):
    return q.get((state, action), 0.0)

def q_update(curr_state, curr_action, next_reward, next_state):
    old_value = get_q(curr_state, curr_action)
    target = next_reward + GAMMA * max(get_q(next_state, a) for a in simulator.get_actions(next_state))
    new_value = old_value + ALPHA * (target - old_value)
    q[(curr_state, curr_action)] = new_value
    return old_value, new_value

q_random_rows = []
for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        old_value, new_value = q_update(curr_state, curr_action, next_reward, next_state)
        q_random_rows.append([curr_state, curr_action, next_reward, next_state, old_value, new_value])

Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [29]:
# YOUR CHANGES HERE

with open("q-random.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
    writer.writerows(q_random_rows)

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [30]:
# YOUR CHANGES HERE

def greedy_action(state, actions):
    best = max(get_q(state, a) for a in actions)
    best_actions = [a for a in actions if get_q(state, a) == best]
    return random.choice(best_actions)

def epsilon_greedy_policy(state, actions):
    if random.random() < 0.25:
        return random.choice(actions)
    return greedy_action(state, actions)

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [31]:
# YOUR CHANGES HERE

q = {}
q_greedy_rows = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        old_value, new_value = q_update(curr_state, curr_action, next_reward, next_state)
        q_greedy_rows.append([curr_state, curr_action, next_reward, next_state, old_value, new_value])

with open("q-greedy.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
    writer.writerows(q_greedy_rows)

print("Rows:", len(q_greedy_rows))

Rows: 32768


Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [32]:
# YOUR CHANGES HERE

def extract_policy():
    num_states = simulator.resolution * simulator.resolution * 8
    policy = {}
    for state in range(num_states):
        actions = simulator.get_actions(state)
        policy[state] = max(actions, key=lambda a: (get_q(state, a), -abs(a)))
    return policy

def extract_policy_visited(states):
    policy = {}
    for state in states:
        actions = simulator.get_actions(state)
        policy[state] = max(actions, key=lambda a: (get_q(state, a), -abs(a)))
    return policy

def save_policy(policy, filename):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(["state", "action"])
        for state in sorted(policy):
            writer.writerow([state, policy[state]])

visited_states = set(row[0] for row in q_greedy_rows)
policy_greedy = extract_policy_visited(visited_states)
save_policy(policy_greedy, "policy-greedy.tsv")

print("States in policy:", len(policy_greedy))

States in policy: 1921


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [33]:
# YOUR CHANGES HERE

q = {}

for episode in range(5000):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        q_update(curr_state, curr_action, next_reward, next_state)
        if next_reward > 0:
            break

num_states = simulator.resolution * simulator.resolution * 8
for sweep in range(200):
    max_change = 0.0
    for state in range(num_states):
        for action in simulator.get_actions(state):
            next_reward, next_state = simulator.get_next_reward_state(state, action)
            old_value, new_value = q_update(state, action, next_reward, next_state)
            max_change = max(max_change, abs(new_value - old_value))
    if max_change < 1e-9:
        break

print("Sweeps until converged:", sweep + 1)

policy_optimal = extract_policy()
save_policy(policy_optimal, "policy-optimal.tsv")

Sweeps until converged: 8


In [34]:
def evaluate_policy(policy, start_state, max_steps=1024):
    state = start_state
    total = 0.0
    for t in range(max_steps):
        reward, state = simulator.get_next_reward_state(state, policy[state])
        total += (GAMMA ** t) * reward
        if reward > 0:
            break
    return total

values = [evaluate_policy(policy_optimal, s) for s in simulator.initial_states]
print("Average value over initial states:", sum(values) / len(values))
print("Initial states that never reach the goal:", sum(v == 0 for v in values))

Average value over initial states: 0.4017391038630001
Initial states that never reach the goal: 0


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.